<img src="icon.png" width=128/>

# Allo: Accelerator Design and Programming Language

Lecture: SEU - FPGA HLS Design

Speaker: Kai Shao

Date: 2026/06/09

# 3 · Functional Simulation

> **FPGA HLS Design** — Allo hands-on series (3/4)

Synthesis is slow (minutes to hours). Before you pay that cost you want to know your
algorithm is **functionally correct**. Allo gives you two simulators, both reached
through the familiar `schedule().export(...)` interface:

| simulator | how | speed | fidelity |
|---|---|---|---|
| **CPU (JIT)** | `export("cpu")` then call it | very fast | bit-accurate for types & arithmetic |
| **Vitis C-sim** | `export("vitis")` then call it | slower (compiles HLS C++) | exactly the C++ that gets synthesized |

Use the **CPU** simulator for everyday correctness checks, and **Vitis C-sim** when you
need to validate the actual HLS design — especially **dataflow / systolic** kernels.

This notebook covers:

1. CPU simulation of plain kernels.
2. **Bit-accurate** arbitrary-precision simulation.
3. **Streams & dataflow** simulation (producer/consumer).
4. An important caveat about the CPU simulator and multi-PE arrays.
5. **Vitis C-simulation** of the systolic GEMM.
6. Stateful variables across calls.

In [ ]:
import numpy as np
import allo.exp as allo
from allo.exp.lang.kernel import kernel
from allo.exp.lang.core import i32, f32, APInt, Stream
from allo.exp.backend.vitis.core import is_vitis_available
print("Vitis available:", is_vitis_available())

# In VSCode/Jupyter, route Allo's logs to plain text instead of a live spinner
# widget. The spinner can render as an empty Output() in VSCode, making csim /
# synthesis look like it never starts (it is actually running underneath).
import allo.exp.logging as _allo_log
from rich.console import Console as _Console
_allo_log.console = _Console(stderr=True, force_interactive=False)

## 3.1 CPU simulation

`export("cpu")` lowers the kernel to LLVM, JIT-compiles it into a shared library, and
returns a plain callable. Arrays are passed as NumPy arrays (in/out by reference). This
is the fastest way to check a design.

In [ ]:
M, N, K = 32, 32, 32

@kernel
def gemm(A: f32[M, K], B: f32[K, N], C: f32[M, N]):
    for i in allo.range(M, name="i"):
        for j in allo.range(N, name="j"):
            for k in allo.range(K, name="k"):
                C[i, j] += A[i, k] * B[k, j]

# A schedule does not change results -- only hardware. Simulate to confirm.
s = gemm.schedule()
s.reorder((s.loop("i"), s.loop("k"), s.loop("j")))
s.pipeline(s.loop("j"), ii=1)
mod = s.export("cpu")

A = np.random.rand(M, K).astype(np.float32)
B = np.random.rand(K, N).astype(np.float32)
C = np.zeros((M, N), dtype=np.float32)
mod(A, B, C)
print("max abs error vs numpy:", float(np.max(np.abs(C - A @ B))))
assert np.allclose(C, A @ B, rtol=1e-4)

## 3.2 Bit-accurate simulation

This is where a *hardware* simulator differs from running plain Python. Allo models the
**exact bit width** of every type: integer arithmetic wraps, narrow types overflow, and
signedness is respected — just like the synthesized circuit.

To make the effect visible we must pick operands whose result actually **leaves** the
type's range. Below, several lanes of `A[i] + B[i]` exceed the 5-bit signed range
`[-16, 15]`, so the hardware result *wraps* — and diverges from the naive full-width add a
CPU would give. Catching this here saves a wrong-but-passes-in-`float` surprise on the
board.

In [ ]:
i5 = APInt(5, signed=True)     # range [-16, 15]
u5 = APInt(5, signed=False)    # range [0, 31]

@kernel
def addsub(A: i5[8], B: u5[8], C: i5[8]):
    for i in allo.range(8, name="i"):
        C[i] = A[i] + B[i]

# Pick lanes whose sum overflows i5 so the 5-bit wrap is actually exercised.
A = np.array([10, 12, 15, -16, -10,  7,  8, 13], dtype=np.int8)   # all within [-16, 15]
B = np.array([10,  8,  1,   0,  20, 25, 30,  5], dtype=np.uint8)  # all within [0, 31]
C = np.zeros(8, dtype=np.int8)
addsub(A, B, C)

true_sum = A.astype(np.int16) + B                                  # what a CPU computes
i5_wrap  = ((true_sum + 16) % 32 - 16).astype(np.int8)            # what 5-bit hw computes
print("true A + B (full width):", true_sum)
print("Allo i5 simulation     :", C)
print("i5 wrap (expected)     :", i5_wrap)
print(f"Allo matches 5-bit hardware: {np.array_equal(C, i5_wrap)};"
      f" differs from the full-width add in {int((C != true_sum).sum())}/8 lanes")
assert np.array_equal(C, i5_wrap)

## 3.3 Streams & dataflow simulation

The dataflow building block is the **`Stream`** — a FIFO between concurrent kernels.
A `producer` writes with `.put(...)` and a `consumer` reads with `.get()`; the CPU
simulator runs them as communicating tasks. This is the simplest form of the dataflow
model that scales up to systolic arrays.

In [ ]:
@kernel
def stream_top(x: i32[8], out: i32[8]):
    fifo: Stream[i32]

    @kernel
    def producer(src: i32[8], s: Stream[i32]):
        for i in range(8):
            s.put(src[i] + 1)

    @kernel
    def consumer(s: Stream[i32], dst: i32[8]):
        for i in range(8):
            dst[i] = s.get() * 2

    producer(x, fifo)
    consumer(fifo, out)

x = np.arange(8, dtype=np.int32)
out = np.zeros(8, dtype=np.int32)
stream_top(x, out)
print("out:", out)
assert np.array_equal(out, (x + 1) * 2)

Streams can also carry **blocks** (`Stream[i32[2, 2]]`) — a whole tile per transfer —
which the backend lowers to a scalar FIFO with a depth scaled by the block size. Block
streams simulate the same way (`.put(buf)` / `buf = .get()`).

## 3.4 Caveat: the CPU simulator and multi-PE arrays

The CPU dataflow simulator handles simple producer/consumer graphs, but it **currently
deadlocks on multi-PE arrays** (e.g. a `mapping=[P0, P1]` systolic grid with many FIFOs
in flight). The hang is an artefact of the CPU scheduling of the tasks, **not** a bug in
your kernel logic.

> **Rule of thumb:** simulate plain kernels and small producer/consumer graphs on the
> **CPU**; validate **systolic / multi-PE** designs with **Vitis C-simulation** (next).

## 3.5 Vitis C-simulation of the systolic GEMM

Calling a **Vitis** backend object actually runs **C-simulation**: it compiles the exact
HLS C++ that will be synthesized and executes it against your NumPy data. This is the
trustworthy check for dataflow kernels.

Below is the **2D output-stationary systolic GEMM** — an `(M+2) × (N+2)` PE grid. The
border PEs feed `A` from the left and `B` from the top; interior PEs multiply-accumulate
and forward operands to their right/bottom neighbours through `Stream` FIFOs. This is the
SPMD spatial model from notebook 1, at full strength.

In [ ]:
M2, N2, K2 = 2, 2, 2
P0, P1 = M2 + 2, N2 + 2

@kernel
def systolic_2d(A: f32[M2, K2], B: f32[K2, N2], C: f32[M2, N2]):
    fifo_A: Stream[f32][P0, P1]
    fifo_B: Stream[f32][P0, P1]

    @kernel(mapping=[P0, P1])
    def pe(A: f32[M2, K2], B: f32[K2, N2], C: f32[M2, N2],
           fifo_A: Stream[f32][P0, P1], fifo_B: Stream[f32][P0, P1]):
        i = allo.get_wid(0)
        j = allo.get_wid(1)
        if (i == 0 or i == M2 + 1) and (j == 0 or j == N2 + 1):
            pass                                   # idle corner
        elif j == 0:                               # left edge: inject A
            for k in range(K2):
                fifo_A[i, j + 1].put(A[i - 1, k])
        elif i == 0:                               # top edge: inject B
            for k in range(K2):
                fifo_B[i + 1, j].put(B[k, j - 1])
        elif i == M2 + 1:                          # bottom drain
            for k in range(K2):
                b: f32 = fifo_B[i, j].get()
        elif j == N2 + 1:                          # right drain
            for k in range(K2):
                a: f32 = fifo_A[i, j].get()
        else:                                      # interior PE: MAC + forward
            c: f32 = 0
            for k in range(K2):
                a: f32 = fifo_A[i, j].get()
                b: f32 = fifo_B[i, j].get()
                c += a * b
                fifo_A[i, j + 1].put(a)
                fifo_B[i + 1, j].put(b)
            C[i - 1, j - 1] = c

    pe(A, B, C, fifo_A, fifo_B)

print("kernel built; PE grid =", P0, "x", P1)

In [ ]:
import tempfile

if is_vitis_available():
    A = np.random.rand(M2, K2).astype(np.float32)
    B = np.random.rand(K2, N2).astype(np.float32)
    C = np.zeros((M2, N2), dtype=np.float32)
    with tempfile.TemporaryDirectory() as proj:
        backend = systolic_2d.schedule().export("vitis", project_path=proj)
        backend(A, B, C)        # <-- runs real Vitis C-simulation
    print("systolic C =\n", C)
    print("numpy  A@B =\n", A @ B)
    print("match:", np.allclose(C, A @ B, atol=1e-5))
else:
    print("Vitis HLS not available -- skipping C-simulation.")
    print("(You can still inspect the generated code: systolic_2d.schedule().export('vitis').hls_code)")

## 3.6 Stateful variables across calls

A `Stateful[T]` variable persists between invocations (it lowers to a file-scope global,
i.e. a register/memory that holds state on the device). Under Vitis C-sim the backend
keeps the compiled `.so` loaded, so repeated calls **accumulate** — handy for modelling
counters or running sums.

In [ ]:
from allo.exp.lang.core import Stateful

@kernel
def running_sum(x: i32) -> i32:
    s: Stateful[i32] = 0
    s = s + x
    return s

if is_vitis_available():
    with tempfile.TemporaryDirectory() as proj:
        acc = running_sum.schedule().export("vitis", project_path=proj)
        print(int(acc(5)), int(acc(10)), int(acc(3)))   # 5, 15, 18  -- state persists
else:
    print("Vitis HLS not available -- skipping stateful C-sim.")

## Recap

* **CPU simulation** (`export("cpu")`) is the fast, bit-accurate correctness check for
  plain kernels and simple stream graphs.
* Arbitrary-precision types simulate **exactly** like hardware (wrap/overflow/signedness).
* **Dataflow** is built from `Stream` FIFOs; the CPU sim handles simple graphs but
  **deadlocks on multi-PE arrays** — use **Vitis C-sim** for systolic designs.
* Calling a **Vitis backend object runs C-simulation** of the real HLS C++; this is how
  we validated the systolic GEMM.

**Next:** [`04_backend_synthesis.ipynb`](04_backend_synthesis.ipynb) — turning the
verified design into an FPGA implementation and reading the synthesis report.